# Model agreement

Audit and summarize LLM agreement columns when they are present in the full-endpoint publication parquet.


In [ ]:
import sys
from pathlib import Path

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "utils").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils import shared_paths as P
from utils.data_analysis_00_dataset_analysis import (
    CATEGORY_PATTERNS,
    MODEL_NAMES,
    contains_pattern,
    load_publications,
    model_agreement_columns,
    normalise_bool,
    normalized_rows,
    output_dirs,
    sample_balanced,
    save_figure,
)

P.bootstrap()


In [ ]:
df = load_publications(P.SHOWCASE_PLUS)
df.shape


In [ ]:
table_dir, figure_dir = output_dirs("00_dataset_analysis_03_model_agreement")
required = model_agreement_columns()
available = sorted(required & set(df.columns))
missing = sorted(required - set(df.columns))

schema_audit = pd.DataFrame(
    {
        "metric": ["required_model_columns", "available_model_columns", "missing_model_columns", "analysis_ready"],
        "value": [len(required), len(available), len(missing), not missing],
        "detail": ["", ", ".join(available), ", ".join(missing), ""],
    }
)
schema_audit.to_csv(table_dir / "model_agreement_schema_audit.csv", index=False)
schema_audit


In [ ]:
if missing:
    print("Model-agreement calculations were not run because the publication endpoint parquet does not contain LLM prediction columns.")
else:
    work = df.copy()
    for model in MODEL_NAMES:
        for suffix in ("parse_ok", "true", "false"):
            work[f"{model}_{suffix}"] = normalise_bool(work[f"{model}_{suffix}"])
    for column in ("three_model_TRUE_agreement", "three_model_FALSE_agreement", "all_three_parsed"):
        work[column] = normalise_bool(work[column])

    rows = []
    for model in MODEL_NAMES:
        parsed = int(work[f"{model}_parse_ok"].sum())
        true_count = int(work[f"{model}_true"].sum())
        rows.append(
            {
                "model": model,
                "parsed": parsed,
                "true": true_count,
                "false": int(work[f"{model}_false"].sum()),
                "true_percent_among_parsed": true_count / max(parsed, 1) * 100,
            }
        )
    model_summary = pd.DataFrame(rows)
    model_summary.to_csv(table_dir / "model_positive_rate_summary.csv", index=False)

    vote_distribution = work["n_true_votes"].value_counts().sort_index().rename_axis("n_true_votes").reset_index(name="n_candidates")
    vote_distribution.to_csv(table_dir / "true_vote_distribution.csv", index=False)

    consensus_distribution = work["consensus_group"].value_counts().rename_axis("consensus_group").reset_index(name="n_candidates")
    consensus_distribution.to_csv(table_dir / "consensus_group_distribution.csv", index=False)

    figure, axis = plt.subplots(figsize=(8.5, 5))
    axis.bar(model_summary["model"], model_summary["true_percent_among_parsed"])
    axis.set(ylabel="TRUE among parsed rows (%)", title="Model-level positive rate")
    axis.grid(axis="y", alpha=0.25)
    save_figure(figure, figure_dir / "model_positive_rate_summary.png")
    model_summary
